In [43]:
from langchain.chat_models import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
import os
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
import pinecone
from pinecone import Pinecone
from langchain.agents import initialize_agent, Tool
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

load_dotenv('/upstage-ai-advanced-ir7/.env')

os.environ["UPSTAGE_API_KEY"] = os.getenv('UPSTAGE_API_KEY')
pinecone_api_key = os.getenv('PINE_API_KEY')

In [44]:
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index("session-chat-index")
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.7, api_key=os.getenv("OPENAI_API_KEY"))

In [45]:
def store_conversation(session_id, text, role):
    # 대화 텍스트를 임베딩으로 변환
    text_embedding = embeddings.embed_query(text)

    # 고유한 벡터 ID 생성 (session ID와 대화 번호를 결합)
    unique_id = f"{session_id}-{role}-{len(text)}"

    # 메타데이터 설정 (role: user 또는 ai)
    metadata = {
        "session_id": session_id,
        "text": text,  # 실제 대화 내용을 메타데이터에 포함
        "role": role  # user or ai
    }

    # Pinecone에 벡터 저장 (Upsert)
    index.upsert(vectors=[(unique_id, text_embedding, metadata)])

# Pinecone에서 대화 검색하는 함수
def retrieve_conversations(session_id, query_text, top_k=5):
    # 쿼리 텍스트를 임베딩으로 변환
    query_embedding = embeddings.embed_query(query_text)

    # session_id를 필터로 사용하여 대화 검색
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        filter={"session_id": {"$eq": session_id}},
        include_metadata=True  # 메타데이터를 포함해서 검색
    )
    for match in results["matches"]:
        metadata = match.get('metadata')
        if metadata:
            return metadata.get('text', 'No text available')
    return "과거 대화를 찾을 수 없습니다."
    # 검색된 결과를 출력
    # for match in results["matches"]:
    #     metadata = match.get('metadata')  # 메타데이터 가져오기
    #     if metadata:
    #         print(f"Role: {metadata.get('role', 'unknown')}")
    #         print(f"Text: {metadata.get('text', 'No text available')}")
    #         print(f"Session ID: {metadata.get('session_id', 'No text available')}")
    #     else:
    #         print("No metadata available")
    #     print(f"Score: {match['score']}")
    #     print()

def search_history_tool(session_id, query_text):
    return retrieve_conversations(session_id, query_text)

In [46]:
# 예시: 대화 저장
store_conversation("session123", "오늘 날씨가 어때?", "user")  # 유저 발화 저장
store_conversation("session123", "오늘은 맑습니다.", "ai")  # AI 응답 저장


In [47]:
# 예시: 대화 검색
retrieve_conversations("session123", "날씨 어때?")

'오늘 날씨가 어때?'

In [48]:
tools = [
    Tool(
        name="SearchHistory",
        func=lambda query: search_history_tool("session123", query),  # 특정 세션에서 검색
        description="Searches the chat history based on the user's question."
    )
]

# Agent 초기화 (에이전트에 도구 전달)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    memory=ConversationBufferMemory(memory_key="chat_history"),
    handle_parsing_errors=True  # 에러 발생 시 재시도 옵션 추가
)

In [49]:
def chatbot(session_id, user_input):
    # 에이전트를 통해 유저의 입력을 처리
    response = agent.run(user_input)

    # 대화를 Pinecone에 저장 (유저 입력 및 AI 응답)
    store_conversation(session_id, user_input, "user")
    store_conversation(session_id, response, "ai")

    return response

In [50]:
session_id = "session123"
print(chatbot(session_id, "내 이름은 허동재야"))

KeyboardInterrupt: 

In [42]:
print(chatbot(session_id, "내 이름이 뭐야?"))

내 이름은 허동재야.


### 새로운 시도

In [87]:
system_message = """
당신은 대화 기록을 기반으로 질문에 답변하는 유용한 도우미입니다.
만약 사용자가 당신이 모르는 것을 물어보면, 저장된 대화 기록을 참고하여 답을 찾습니다.
항상 정중하고 간결하게 응답하세요.
"""

In [88]:
prompt_template = PromptTemplate.from_template(
    system_message + "\n\nUser: {input}\nAssistant:"
)

In [53]:
tools = [
    Tool(
        name="SearchHistory",
        func=lambda query: search_history_tool("session123", query),  # 세션에 맞춰 검색
        description="Searches the chat history based on the user's question."
    )
]

# 에이전트 초기화
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    memory=ConversationBufferMemory(memory_key="chat_history"),
    verbose=True,  # 디버깅을 위해 상세 로그 추가
    handle_parsing_errors=True
)

In [54]:
def chatbot(session_id, user_input):
    # 에이전트를 통해 유저의 입력을 처리
    response = agent.run(user_input)

    # 대화를 Pinecone에 저장 (유저 입력 및 AI 응답)
    store_conversation(session_id, user_input, "user")
    store_conversation(session_id, response, "ai")

    return response

In [55]:
session_id = "session124"
print(chatbot(session_id, "내 이름은 허동재야"))



> Entering new AgentExecutor chain...
The user is providing their name in Korean. There is no question to answer, but I will acknowledge the information.
Final Answer: 안녕하세요, 허동재님! 만나서 반갑습니다. (Hello, Huh Dong-jae! Nice to meet you.)

> Finished chain.
안녕하세요, 허동재님! 만나서 반갑습니다. (Hello, Huh Dong-jae! Nice to meet you.)


In [56]:
print(chatbot(session_id, "내 이름이 뭐야?"))



> Entering new AgentExecutor chain...
Action: SearchHistory
Action Input: "내 이름이 뭐야?"
Observation: 내 이름이 뭐야?
Thought:Action: SearchHistory
Action Input: "내 이름"
Observation: 내 이름이 뭐야?
Thought:Action: SearchHistory
Action Input: "이름"
Observation: 내 이름이 뭐야?
Thought:Action: SearchHistory
Action Input: "이름이"
Observation: 내 이름이 뭐야?
Thought:Action: SearchHistory
Action Input: "내 이름이 뭐야"
Observation: 내 이름이 뭐야?
Thought:It seems there is no information in the chat history regarding your name. Therefore, I do not know your name. If you provide it, I can remember it for our current conversation.
Observation: Invalid Format: Missing 'Action:' after 'Thought:
Thought:It seems there is no information in the chat history regarding your name. Therefore, I do not know your name. If you provide it, I can remember it for our current conversation.
Observation: Invalid Format: Missing 'Action:' after 'Thought:
Thought:I do not know your name based on the current chat history. If you provide it, I can reme

In [98]:
# 히스토리 검색 함수
def retrieve_conversations(session_id, query_text, top_k=5):
    query_embedding = embeddings.embed_query(query_text)
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        filter={"session_id": {"$eq": session_id}},
        include_metadata=True
    )

    print(f"Query Results for session: {session_id}")
    for match in results["matches"]:
        print(f"Match score: {match['score']} - Text: {match['metadata'].get('text')}")

    if results["matches"]:
        for match in results["matches"]:
            metadata = match.get('metadata')
            if metadata:
                return metadata.get('text', 'No text available')
    return "No relevant information found in the chat history."

from langchain.memory import ConversationBufferMemory

# 세션 ID에 따라 고유한 memory_key 설정
def create_memory_for_session(session_id):
    memory_key = f"chat_history_{session_id}"  # 세션 ID를 포함한 고유한 memory_key 생성
    return ConversationBufferMemory(memory_key=memory_key, k=3)  # 최근 3개 대화만 유지

session_id = "session145"
memory = create_memory_for_session(session_id)

# 검색 도구에 대한 응답 처리 함수
def search_history_tool(session_id, query_text):
    print(f"Searching history for session: {session_id}, query: {query_text}")  # 세션 ID 및 쿼리 로그 추가
    result = retrieve_conversations(session_id, query_text)
    if result:
        return result
    else:
        return "No relevant information found in the chat history."


# Tool을 통한 검색 정의
# Tool을 통해 세션 ID는 별도 처리, 사용자 입력만 사용
tools = [
    Tool(
        name="SearchHistory",
        func=lambda query: search_history_tool(session_id, query),  # session_id는 외부에서 전달
        description="Searches the chat history based on the user's question."
    )
]

# 에이전트 실행 (한 번만 검색하도록 개선)
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    memory=memory,  # 세션별로 설정된 메모리 사용
    verbose=True,
    handle_parsing_errors=True
)

# 챗봇 함수 (한 번만 검색 후 결과 처리)
def chatbot(session_id, user_input):
    # 에이전트에게는 사용자 입력만 전달
    response = agent.run(user_input)  # 오직 user_input만 전달

    # 대화를 Pinecone에 저장 (유저 입력 및 AI 응답)
    store_conversation(session_id, user_input, "user")
    store_conversation(session_id, response, "ai")

    return response

# 테스트 실행


In [90]:

print(chatbot(session_id, "나의 이름은 허동재"))




> Entering new AgentExecutor chain...
To answer the question accurately, I need to determine if the user's name has been mentioned previously in the chat history. 

Action: SearchHistory
Action Input: "나의 이름은 허동재"Searching history for session: session138, query: 나의 이름은 허동재
Query Results for session: session138

Observation: No relevant information found in the chat history.
Thought:I don't have any previous information about your name in the chat history. 

Final Answer: Your name is 허동재.

> Finished chain.
Your name is 허동재.


In [91]:
print(chatbot(session_id, "나의 이름은?"))



> Entering new AgentExecutor chain...
Action: SearchHistory
Action Input: "나의 이름은?"Searching history for session: session138, query: 나의 이름은?
Query Results for session: session138
Match score: 0.654208541 - Text: 나의 이름은 허동재
Match score: 0.556248128 - Text: Your name is 허동재.

Observation: 나의 이름은 허동재
Thought:I now know the final answer.
Final Answer: 나의 이름은 허동재입니다.

> Finished chain.
나의 이름은 허동재입니다.


In [92]:
session_id = "session140"

In [99]:
print(chatbot(session_id, "한국에서 유명한 음식 뭐가 있지?"))



> Entering new AgentExecutor chain...
한국에서 유명한 음식을 찾기 위해 검색 기록을 확인할 필요가 있는지 판단해보겠습니다. 

한국은 다양한 전통 음식으로 유명하며, 그중에서도 몇 가지 대표적인 음식을 말씀드리겠습니다. 

Final Answer: 한국에서 유명한 음식으로는 김치, 불고기, 비빔밥, 삼겹살, 떡볶이, 김밥, 잡채, 갈비, 된장찌개, 그리고 해물파전 등이 있습니다.

> Finished chain.
한국에서 유명한 음식으로는 김치, 불고기, 비빔밥, 삼겹살, 떡볶이, 김밥, 잡채, 갈비, 된장찌개, 그리고 해물파전 등이 있습니다.


In [100]:
print(chatbot(session_id, "순대국알아? 내가 가장 좋아하는 음식인데?"))



> Entering new AgentExecutor chain...
The question is asking if I am familiar with "순대국," which is a dish that the user mentions as their favorite. I need to determine if there is any prior context or information in the chat history about this dish. 

Action: SearchHistory
Action Input: "순대국"Searching history for session: session145, query: 순대국
Query Results for session: session145
Match score: 0.470771223 - Text: 한국에서 유명한 음식 뭐가 있지?
Match score: 0.382772386 - Text: 한국에서 유명한 음식으로는 김치, 불고기, 비빔밥, 삼겹살, 떡볶이, 김밥, 잡채, 갈비, 된장찌개, 그리고 해물파전 등이 있습니다.

Observation: 한국에서 유명한 음식 뭐가 있지?
Thought:It seems that there is a previous query about famous foods in Korea. "순대국" (Sundae-guk) is indeed a popular Korean dish. It is a soup made with Korean blood sausage (sundae) and various other ingredients. I can provide more information about it without needing further context from the chat history. 

Final Answer: Yes, I know about "순대국." It's a popular Korean soup that features blood sausage as one of its ma

In [101]:
print(chatbot(session_id, "어떻게 만드는지 알아?"))



> Entering new AgentExecutor chain...
Action: SearchHistory
Action Input: "어떻게 만드는지 알아"Searching history for session: session145, query: 어떻게 만드는지 알아
Query Results for session: session145
Match score: 0.435916126 - Text: 순대국알아? 내가 가장 좋아하는 음식인데?
Match score: 0.389506698 - Text: 한국에서 유명한 음식 뭐가 있지?
Match score: 0.309003592 - Text: Yes, I know about "순대국." It's a popular Korean soup that features blood sausage as one of its main ingredients. It's often enjoyed for its rich, savory flavor and is a favorite comfort food for many. I'm glad to hear it's your favorite dish!
Match score: 0.281305403 - Text: 한국에서 유명한 음식으로는 김치, 불고기, 비빔밥, 삼겹살, 떡볶이, 김밥, 잡채, 갈비, 된장찌개, 그리고 해물파전 등이 있습니다.

Observation: 순대국알아? 내가 가장 좋아하는 음식인데?
Thought:Action: SearchHistory
Action Input: "어떻게 만드는지 알아"Searching history for session: session145, query: 어떻게 만드는지 알아
Query Results for session: session145
Match score: 0.435916126 - Text: 순대국알아? 내가 가장 좋아하는 음식인데?
Match score: 0.389506698 - Text: 한국에서 유명한 음식 뭐가 있지?
Match score: 0.

In [102]:
print(chatbot(session_id, "내가 가장 좋아하는 음식이 뭐지?"))



> Entering new AgentExecutor chain...
Action: SearchHistory
Action Input: "가장 좋아하는 음식"Searching history for session: session145, query: 가장 좋아하는 음식
Query Results for session: session145
Match score: 0.714185238 - Text: 순대국알아? 내가 가장 좋아하는 음식인데?
Match score: 0.620345 - Text: 한국에서 유명한 음식 뭐가 있지?
Match score: 0.505578518 - Text: Yes, I know about "순대국." It's a popular Korean soup that features blood sausage as one of its main ingredients. It's often enjoyed for its rich, savory flavor and is a favorite comfort food for many. I'm glad to hear it's your favorite dish!
Match score: 0.499963939 - Text: 한국에서 유명한 음식으로는 김치, 불고기, 비빔밥, 삼겹살, 떡볶이, 김밥, 잡채, 갈비, 된장찌개, 그리고 해물파전 등이 있습니다.
Match score: 0.449119717 - Text: 어떻게 만드는지 알아?

Observation: 순대국알아? 내가 가장 좋아하는 음식인데?
Thought:I now know the final answer
Final Answer: 당신이 가장 좋아하는 음식은 순대국입니다.

> Finished chain.
당신이 가장 좋아하는 음식은 순대국입니다.
